In [1]:
from openai import OpenAI
import re
import sys
import os
import json
import time
from tqdm import tqdm

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# %cd drive/My\ Drive/Colab\ Notebooks/Project_folder/FullPipeLine

In [2]:
with open("./../Translation/config.json") as f:
    config_data = json.loads(f.read())



OPENAI_KEY = config_data['OPENAI_KEY']
client = OpenAI(api_key=OPENAI_KEY)

model_name = "gpt-4o-mini-2024-07-18"

In [ ]:
language_translation_codes = [
    ('Acehnese', 'ace_Arab'),
    ('Hebrew', 'heb_Hebr'),
    ('Vietnamese', 'vie_Latn'),
    ('Indonesian', 'ind_Latn'),
    ('Malay', 'mal_Mlym'),
    ('Tagalog', 'tgl_Latn'),
    ('English', 'eng_Latn'),
    ('Dutch', 'nld_Latn'),
    ('German', 'deu_Latn'),
    ('Afrikaans', 'afr_Latn'),
    ('Portuguese', 'por_Latn'),
    ('Spanish', 'spa_Latn'),
    ('French', 'fra_Latn'),
    ('Italian', 'ita_Latn'),
    ('Greek', 'ell_Grek'),
    ('Western Persian', 'pes_Arab'),
    ('Russian', 'rus_Cyrl'),
    ('Bulgarian', 'bul_Cyrl'),
    ('Chinese', 'zho_Hans'),
    ('Turkish', 'tur_Latn'),
    ('Estonian', 'est_Latn'),
    ('Finnish', 'fin_Latn'),
    ('Spanish', 'spa_Latn'),
    ('Hungarian', 'hun_Latn'),
]


In [4]:
def gpt_4o_mini_response(prompt, from_lang, to_lang, temperature, max_tokens):
    # print("Translating from "+from_lang+" to "+to_lang)
    try:
        system_message = f"You are a translator. Translate the following sentence from {from_lang} to {to_lang}.\n"

        response = client.chat.completions.create(
            model=model_name,
            messages=[
                {
                    "role": "system",
                    "content": system_message
                },
                {"role": "user", "content": prompt["prompt"]}
            ],
            temperature=temperature,
            max_tokens=max_tokens,
            n=10,
        )
        # Make a list of all the responses
        responses = []
        for choice in response.choices:
            responses.append(choice.message.content)
        return responses
    except Exception as e:
        print(e)
        return 'Problem occurred.'

In [5]:
gpt_4o_mini_response({"prompt": "How are you?"}, 'English', 'Bangla', 0.7, 50)

['আপনি কেমন আছেন?',
 'আপনি কেমন আছেন?',
 'আপনি কেমন আছেন?',
 'আপনি কেমন আছেন?',
 'আপনি কেমন আছেন?',
 'আপনি কেমন আছেন?',
 'আপনি কেমন আছেন?',
 'আপনি কেমন আছেন?',
 'আপনি কেমন আছেন?',
 'তুমি কেমন আছো?']

In [6]:
def read_json_file(file_path):
    print("File reading starts")

    with open(file_path, 'r') as infile:
        json_data = json.load(infile)  # Use json.load for reading standard JSON

    print("File reading ends")
    return json_data


In [7]:
def read_jsonl_file(file_path):
    print("File reading starts")

    with open(file_path, 'r') as infile:
        lines = infile.readlines()

    json_data = [json.loads(line) for line in lines]
    print("File reading ends")

    return json_data

In [8]:
def process_translations(json_data, key):
    print("Processing translations...")

    new_columnName = 'translations'
    updated_data = []

    for data in tqdm(json_data):
        if key in data:
            key_value = data[key]
            # print("Key_value: ",key_value)

            translations = {}
            for language, code in language_translation_codes:
                translations[language] = gpt_4o_mini_response(
                    {"prompt": key_value},
                    'English',
                    language,
                    0.7,
                    512
                )
                # print("Language: "+language)
                # print(translations[language])
        else:
            # In case the key is not found, keep the translations empty
            translations = {language: "" for language, _ in language_translation_codes}

        # Add the translations dictionary as a new field in the data
        data[new_columnName] = translations
        updated_data.append(data)

    print("Processed translations")
    return updated_data

In [9]:
def save_processed_data(json_data, new_filePath):
    # if os.path.exists(new_filePath):
    #     print(new_filePath+" already exists. No need to proceed.")
    #     return

    print("Saving processed data to "+new_filePath)
    with open(new_filePath, 'w', encoding='utf-8') as outfile:
        for data in json_data:
            outfile.write(json.dumps(data,ensure_ascii=False) + '\n')

    print("Processing completed. Data saved to "+new_filePath)


In [10]:
def process_file(file_path, key):
    base_filename = os.path.splitext(file_path)[0]
    extension = os.path.splitext(file_path)[-1].lower()

    print("Base name: "+base_filename+"Extension: "+extension)

    new_filePath = base_filename + '_gpt_translated_prompt' + extension
    print("New file name: "+new_filePath)

    if extension != '.jsonl' and extension != '.json':
        print("Unsupported file extension")


    if extension == '.jsonl':
        json_data = read_jsonl_file(file_path)
    else:
        json_data = read_json_file(file_path)

    processed_data = process_translations(json_data, key)
    save_processed_data(processed_data, new_filePath)

In [ ]:
current_path = os.getcwd()

# Print it
print(f"Current working directory: {current_path}")
file_path = "./../Dataset/sallm_nl_prompt.jsonl"
key = "prompt_nl_prompt"

# Convert to absolute path to ensure correctness
file_path = os.path.abspath(file_path)

if not os.path.exists(file_path):
    print(f"Error: File {file_path} does not exist.")
    sys.exit(1)

print("file path: "+file_path+", key: "+key)

print("Starting file processing")
process_file(file_path, key)

Current working directory: /Users/lsiddiqsunny/Documents/Notre_Dame/Research/multiNL-SE-Benchmarking/Translation
file path: /Users/lsiddiqsunny/Documents/Notre_Dame/Research/multiNL-SE-Benchmarking/Translation/ProcessedFiles/sallm_nl_prompt.jsonl, key: prompt_nl_prompt
Starting file processing
Base name: /Users/lsiddiqsunny/Documents/Notre_Dame/Research/multiNL-SE-Benchmarking/Translation/ProcessedFiles/sallm_nl_promptExtension: .jsonl
New file name: /Users/lsiddiqsunny/Documents/Notre_Dame/Research/multiNL-SE-Benchmarking/Translation/ProcessedFiles/sallm_nl_prompt_gpt_translated_prompt.jsonl
File reading starts
File reading ends
Processing translations...


100%|██████████| 100/100 [1:29:35<00:00, 53.76s/it]

Processed translations
Saving processed data to /Users/lsiddiqsunny/Documents/Notre_Dame/Research/multiNL-SE-Benchmarking/Translation/ProcessedFiles/sallm_nl_prompt_gpt_translated_prompt.jsonl
Processing completed. Data saved to /Users/lsiddiqsunny/Documents/Notre_Dame/Research/multiNL-SE-Benchmarking/Translation/ProcessedFiles/sallm_nl_prompt_gpt_translated_prompt.jsonl
